            # Plot DeepMD vs DFT correlation metrics

            This notebook reads the text files written by `dp_inference.ipynb` and produces parity plots,
            error histograms, and a text summary of regression statistics.

            Run this notebook from the DeepMD Python environment.
            


In [ ]:
            from pathlib import Path

            import matplotlib.pyplot as plt
            import numpy as np
            import pandas as pd
            


            ## Configuration

            `STYLE_FILE` is optional. Leave it as `None` to use the default Matplotlib style.
            


In [ ]:
            PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "model_analysis" else Path.cwd().resolve()

            INFERENCE_DIR = PROJECT_ROOT / "model_analysis" / "outputs" / "inference"
            STYLE_FILE = None
            N_ATOMS = None

            ENERGY_FILE = INFERENCE_DIR / "energies_forces.txt"
            ATOM_FILE = INFERENCE_DIR / "energies_forces_atoms.txt"
            REPORT_FILE = PROJECT_ROOT / "model_analysis" / "outputs" / "dp_vs_dft_stats.txt"

            print(f"Energy file: {ENERGY_FILE}")
            print(f"Atom file: {ATOM_FILE}")
            print(f"Report file: {REPORT_FILE}")
            


In [ ]:
            if STYLE_FILE is not None:
                plt.style.use(STYLE_FILE)

            if not ENERGY_FILE.exists() or not ATOM_FILE.exists():
                raise FileNotFoundError(
                    "Inference outputs were not found. Run dp_inference.ipynb first or update the paths in the configuration cell."
                )

            energy_df = pd.read_csv(
                ENERGY_FILE,
                comment="#",
                delim_whitespace=True,
                names=["frame", "energy_DFT_eV", "energy_DP_eV"],
            )

            atom_df = pd.read_csv(
                ATOM_FILE,
                comment="#",
                delim_whitespace=True,
                names=[
                    "frame",
                    "atom",
                    "element",
                    "fx_DFT",
                    "fy_DFT",
                    "fz_DFT",
                    "fx_DP",
                    "fy_DP",
                    "fz_DP",
                    "energy_DFT",
                    "energy_DP",
                ],
            )

            energy_df = energy_df.dropna(subset=["energy_DFT_eV", "energy_DP_eV"]).reset_index(drop=True)
            atom_df = atom_df.dropna(
                subset=["fx_DFT", "fy_DFT", "fz_DFT", "fx_DP", "fy_DP", "fz_DP"]
            ).reset_index(drop=True)

            print(f"Loaded {len(energy_df)} frame-level entries")
            print(f"Loaded {len(atom_df)} atom-level entries")
            


In [ ]:
            def stats(y_true, y_pred):
                diff = np.asarray(y_pred) - np.asarray(y_true)
                mae = np.mean(np.abs(diff))
                rmse = np.sqrt(np.mean(diff ** 2))
                r2 = 1.0 - np.sum(diff ** 2) / np.sum((y_true - np.mean(y_true)) ** 2) if len(y_true) > 1 else np.nan
                return mae, rmse, r2
            


            ## Energy parity and error distributions
            


In [ ]:
            y_true = energy_df["energy_DFT_eV"].to_numpy()
            y_pred = energy_df["energy_DP_eV"].to_numpy()
            _, rmse, r2 = stats(y_true, y_pred)

            lims = [min(y_true.min(), y_pred.min()) - 0.1, max(y_true.max(), y_pred.max()) + 0.1]
            abs_err = np.abs(y_pred - y_true)

            fig, ax = plt.subplots(figsize=(6, 6))
            scatter = ax.scatter(
                y_true,
                y_pred,
                c=abs_err,
                s=20,
                cmap="viridis",
                alpha=0.9,
                edgecolors="none",
            )
            ax.plot(lims, lims, color="black", linewidth=1)
            ax.set_xlim(lims)
            ax.set_ylim(lims)
            ax.set_aspect("equal", adjustable="box")
            ax.set_xlabel("Energy DFT (eV)")
            ax.set_ylabel("Energy DP (eV)")
            ax.text(
                0.05,
                0.95,
                f"RMSE = {rmse:.4f} eV\nR² = {r2:.4f}",
                transform=ax.transAxes,
                fontsize=12,
                verticalalignment="top",
            )
            cbar = fig.colorbar(scatter, ax=ax, pad=0.03)
            cbar.set_label("Absolute error (eV)")
            fig.tight_layout()
            plt.show()
            


In [ ]:
            if N_ATOMS is None:
                atom_counts = atom_df.groupby("frame").size().reindex(energy_df["frame"]).to_numpy(dtype=float)
            else:
                atom_counts = np.full(len(energy_df), float(N_ATOMS))

            valid_atom_counts = np.isfinite(atom_counts) & (atom_counts > 0)
            y_true_pa = y_true[valid_atom_counts] / atom_counts[valid_atom_counts]
            y_pred_pa = y_pred[valid_atom_counts] / atom_counts[valid_atom_counts]

            _, rmse_pa, r2_pa = stats(y_true_pa, y_pred_pa)
            abs_err_pa = np.abs(y_pred_pa - y_true_pa) * 1000.0

            lims_pa = [
                min(y_true_pa.min(), y_pred_pa.min()) - 0.01,
                max(y_true_pa.max(), y_pred_pa.max()) + 0.01,
            ]

            fig, ax = plt.subplots(figsize=(6, 6))
            scatter = ax.scatter(
                y_true_pa,
                y_pred_pa,
                c=abs_err_pa,
                s=20,
                cmap="viridis",
                alpha=0.9,
                edgecolors="none",
            )
            ax.plot(lims_pa, lims_pa, color="black", linewidth=1)
            ax.set_xlim(lims_pa)
            ax.set_ylim(lims_pa)
            ax.set_aspect("equal", adjustable="box")
            ax.set_xlabel("Energy/atom DFT (eV/atom)")
            ax.set_ylabel("Energy/atom DP (eV/atom)")
            ax.text(
                0.05,
                0.95,
                f"RMSE = {rmse_pa * 1000:.2f} meV/atom\nR² = {r2_pa:.4f}",
                transform=ax.transAxes,
                fontsize=12,
                verticalalignment="top",
            )
            cbar = fig.colorbar(scatter, ax=ax, pad=0.03)
            cbar.set_label("Absolute error (meV/atom)")
            fig.tight_layout()
            plt.show()
            


In [ ]:
            errors = y_pred - y_true

            fig, ax = plt.subplots(figsize=(6, 4))
            ax.hist(errors, bins=80, color="steelblue", alpha=0.8, edgecolor="black")
            ax.axvline(0.0, color="black", linestyle="--")
            ax.set_xlabel("Energy error (DP - DFT) [eV]")
            ax.set_ylabel("Count")
            ax.set_title(
                f"Mean = {errors.mean():.4f} eV, Std = {errors.std():.4f} eV"
            )
            fig.tight_layout()
            plt.show()

            errors_pa = y_pred_pa - y_true_pa
            fig, ax = plt.subplots(figsize=(6, 4))
            ax.hist(errors_pa, bins=80, color="steelblue", alpha=0.8, edgecolor="black")
            ax.axvline(0.0, color="black", linestyle="--")
            ax.set_xlabel("Energy/atom error (DP - DFT) [eV/atom]")
            ax.set_ylabel("Count")
            ax.set_title(
                f"Mean = {errors_pa.mean():.6f} eV/atom, Std = {errors_pa.std():.6f} eV/atom"
            )
            fig.tight_layout()
            plt.show()
            


            ## Force parity plots
            


In [ ]:
            for component in ["x", "y", "z"]:
                y_true_force = atom_df[f"f{component}_DFT"].to_numpy()
                y_pred_force = atom_df[f"f{component}_DP"].to_numpy()
                mae, rmse, r2 = stats(y_true_force, y_pred_force)

                force_limit = max(np.max(np.abs(y_true_force)), np.max(np.abs(y_pred_force)))
                fig, ax = plt.subplots(figsize=(6, 6))
                heatmap = ax.hexbin(y_true_force, y_pred_force, gridsize=120, bins="log", cmap="viridis")
                ax.plot([-force_limit, force_limit], [-force_limit, force_limit], color="black", linewidth=1)
                ax.set_xlim([-force_limit, force_limit])
                ax.set_ylim([-force_limit, force_limit])
                ax.set_aspect("equal", adjustable="box")
                ax.set_xlabel(f"F{component} DFT (eV/Å)")
                ax.set_ylabel(f"F{component} DP (eV/Å)")
                ax.set_title(f"Force {component.upper()} parity")
                ax.text(
                    0.05,
                    0.95,
                    f"MAE = {mae:.4f}\nRMSE = {rmse:.4f}\nR² = {r2:.4f}",
                    transform=ax.transAxes,
                    fontsize=11,
                    verticalalignment="top",
                    bbox=dict(boxstyle="round", facecolor="white", alpha=0.9, edgecolor="0.8"),
                )
                fig.colorbar(heatmap, ax=ax, label="log(count)", pad=0.03)
                fig.tight_layout()
                plt.show()
            


In [ ]:
            force_ref = np.sqrt(atom_df["fx_DFT"] ** 2 + atom_df["fy_DFT"] ** 2 + atom_df["fz_DFT"] ** 2).to_numpy()
            force_dp = np.sqrt(atom_df["fx_DP"] ** 2 + atom_df["fy_DP"] ** 2 + atom_df["fz_DP"] ** 2).to_numpy()
            mae, rmse, r2 = stats(force_ref, force_dp)

            force_limit = max(np.max(np.abs(force_ref)), np.max(np.abs(force_dp))) + 0.5
            fig, ax = plt.subplots(figsize=(6, 6))
            heatmap = ax.hexbin(force_ref, force_dp, gridsize=120, bins="log", cmap="viridis")
            ax.plot([0.0, force_limit], [0.0, force_limit], color="black", linewidth=1)
            ax.set_xlim([0.0, force_limit])
            ax.set_ylim([0.0, force_limit])
            ax.set_aspect("equal", adjustable="box")
            ax.set_xlabel("|F| DFT (eV/Å)")
            ax.set_ylabel("|F| DP (eV/Å)")
            ax.set_title("Force magnitude parity")
            ax.text(
                0.05,
                0.95,
                f"MAE = {mae:.4f}\nRMSE = {rmse:.4f}\nR² = {r2:.4f}",
                transform=ax.transAxes,
                fontsize=11,
                verticalalignment="top",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.9, edgecolor="0.8"),
            )
            fig.colorbar(heatmap, ax=ax, label="log(count)", pad=0.03)
            fig.tight_layout()
            plt.show()
            


            ## Force-angle difference

            This plot shows the angular mismatch between the DFT and DeepMD force vectors for each atom.
            


In [ ]:
            force_ref_vectors = atom_df[["fx_DFT", "fy_DFT", "fz_DFT"]].to_numpy(float)
            force_dp_vectors = atom_df[["fx_DP", "fy_DP", "fz_DP"]].to_numpy(float)

            norm_ref = np.linalg.norm(force_ref_vectors, axis=1)
            norm_dp = np.linalg.norm(force_dp_vectors, axis=1)
            mask = (norm_ref > 1e-12) & (norm_dp > 1e-12)

            force_ref_vectors = force_ref_vectors[mask]
            force_dp_vectors = force_dp_vectors[mask]
            norm_ref = norm_ref[mask]
            norm_dp = norm_dp[mask]

            cosine = np.sum(force_ref_vectors * force_dp_vectors, axis=1) / (norm_ref * norm_dp)
            cosine = np.clip(cosine, -1.0, 1.0)
            angle_difference = np.degrees(np.arccos(cosine))
            


In [ ]:
            fig, ax = plt.subplots(figsize=(7, 4.5))
            ax.hist(angle_difference, bins=90, color="steelblue", alpha=0.85, edgecolor="black")
            ax.axvline(angle_difference.mean(), color="darkred", linestyle="--", linewidth=1.5)
            ax.set_xlabel("Angle difference between DFT and DP force vectors (deg)")
            ax.set_ylabel("Count")
            ax.set_title("Distribution of force-vector angle differences")
            ax.text(
                0.98,
                0.95,
                (
                    f"Mean = {angle_difference.mean():.2f} deg\n"
                    f"Median = {np.median(angle_difference):.2f} deg\n"
                    f"P90 = {np.percentile(angle_difference, 90):.2f} deg"
                ),
                transform=ax.transAxes,
                ha="right",
                va="top",
                fontsize=11,
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.9, edgecolor="0.8"),
            )
            fig.tight_layout()
            plt.show()
            


            ## Export summary statistics
            


In [ ]:
            REPORT_FILE.parent.mkdir(parents=True, exist_ok=True)

            with REPORT_FILE.open("w", encoding="utf-8") as handle:
                handle.write("=== DeePMD vs DFT Statistics Report ===\n\n")

                energy_ref = energy_df["energy_DFT_eV"].to_numpy(float)
                energy_dp = energy_df["energy_DP_eV"].to_numpy(float)
                energy_mask = np.isfinite(energy_ref) & np.isfinite(energy_dp)
                energy_ref = energy_ref[energy_mask]
                energy_dp = energy_dp[energy_mask]
                energy_error = energy_dp - energy_ref
                mae, rmse, r2 = stats(energy_ref, energy_dp)
                handle.write(
                    "[Energies]\n"
                    f"N samples   : {energy_ref.size}\n"
                    f"MAE         : {mae:.6f} eV\n"
                    f"RMSE        : {rmse:.6f} eV\n"
                    f"R²          : {r2:.6f}\n"
                    f"Mean error  : {energy_error.mean():.6f} eV\n"
                    f"Std error   : {energy_error.std():.6f} eV\n"
                    f"Min/Max err : {energy_error.min():.6f} / {energy_error.max():.6f} eV\n\n"
                )

                for component in ["x", "y", "z"]:
                    force_ref_component = atom_df[f"f{component}_DFT"].to_numpy(float)
                    force_dp_component = atom_df[f"f{component}_DP"].to_numpy(float)
                    component_mask = np.isfinite(force_ref_component) & np.isfinite(force_dp_component)
                    force_ref_component = force_ref_component[component_mask]
                    force_dp_component = force_dp_component[component_mask]
                    force_error = force_dp_component - force_ref_component
                    mae, rmse, r2 = stats(force_ref_component, force_dp_component)
                    handle.write(
                        f"[Forces {component.upper()}]\n"
                        f"N samples   : {force_ref_component.size}\n"
                        f"MAE         : {mae:.6f} eV/Å\n"
                        f"RMSE        : {rmse:.6f} eV/Å\n"
                        f"R²          : {r2:.6f}\n"
                        f"Mean error  : {force_error.mean():.6f} eV/Å\n"
                        f"Std error   : {force_error.std():.6f} eV/Å\n"
                        f"Min/Max err : {force_error.min():.6f} / {force_error.max():.6f} eV/Å\n\n"
                    )

                force_error = force_dp - force_ref
                mae, rmse, r2 = stats(force_ref, force_dp)
                handle.write(
                    "[Force Magnitudes]\n"
                    f"N samples   : {force_ref.size}\n"
                    f"MAE         : {mae:.6f} eV/Å\n"
                    f"RMSE        : {rmse:.6f} eV/Å\n"
                    f"R²          : {r2:.6f}\n"
                    f"Mean error  : {force_error.mean():.6f} eV/Å\n"
                    f"Std error   : {force_error.std():.6f} eV/Å\n"
                    f"Min/Max err : {force_error.min():.6f} / {force_error.max():.6f} eV/Å\n\n"
                )

                handle.write(
                    "[Force Directions]\n"
                    f"N samples   : {angle_difference.size}\n"
                    f"Mean angle  : {angle_difference.mean():.6f} deg\n"
                    f"Std angle   : {angle_difference.std():.6f} deg\n"
                    f"Median      : {np.median(angle_difference):.6f} deg\n"
                    f"P90         : {np.percentile(angle_difference, 90):.6f} deg\n"
                    f"Min/Max     : {angle_difference.min():.6f} / {angle_difference.max():.6f} deg\n"
                )

            print(f"Saved statistics report to {REPORT_FILE}")
            
